# 56-bus Main Performance Comparison

This notebook runs only the main 5000-scenario comparison for Linear, Safe-DDPG, and RLC-FT.

In [1]:
from Environment import *
from DDPG import *
from NN_Module import *
import os
from config import *
import sys

import torch
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from numpy import linalg as LA

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from pandapower.plotting.plotly import pf_res_plotly

from loguru import logger

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_default_device(device)
print(f"Using device: {device}")

### a simple logger
logger.remove()
logger.add(sys.stderr, level='INFO')


Using device: cuda


4

In [2]:
env_seed = 7        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l
episode_num = 5000   # the total test episode
step_num = 200      # the longest test step

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
state, topology, senario = env.reset_topo(seed=env_seed)
topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
# pf_res_plotly(pp_net);

D:\Code\Python\Flexible_Voltage_Control\.venv\Lib\site-packages\pandapower\converter\pypower\from_ppc.py:334: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  branch_lookup.loc[is_trafo, "element"] = idx_trafo


### Load model

In [3]:
agent_num = 5
agent_policy_net = []
safe_agent_net = []

### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=256)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=2048).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-07-05/Step_300_Seed_45_a{i}.pth')
    # policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth')
    # policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_600_Seed_12_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'), map_location=device)

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth', map_location=device)
    safe_agent_net[i].load_state_dict(policy_net_dict)

### Flexible NN Contoller

In [4]:
### test our controller
voltage = []
q = []
cost = []
success_list = []
fail_list = []
entire_list = []
control_cost = []
reward_list = []
object_cost = []
voltage_trajs = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    cost = []
    abnormal_stop = False
    done = False

    voltage_episode = []   # stores full voltage vectors for this episode

    for step in range(step_num):
        action = []
        for i in range(agent_num):
            action_agent = agent_policy_net[i](torch.tensor(state[i].reshape(1,), dtype=torch.float32, device=device).unsqueeze(0), topology)
            action_agent = 0.7 * action_agent.detach().cpu().numpy()[0] #0.7
            action.append(action_agent)

        action = last_action - np.asarray(action)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            success_list.append((episode,step))
            # Suppress per-episode success logs for large notebook runs.
            break

        voltage.append(state)
        voltage_episode.append(state.copy())  # record full state vector

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    reward_list.append(episode_reward)
    control_cost.append(episode_control)
    object_cost.append(np.sum(cost))
    if len(voltage_episode) > 0 and senario == 0:
        voltage_trajs.append(np.vstack(voltage_episode))

    if (not done) and (abnormal_stop == False):
        entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

    if (episode + 1) % 100 == 0 or (episode + 1) == episode_num:
        logger.info('RLC-FT progress: {}/{} episodes, success={}, fail={}', episode + 1, episode_num, len(success_list), len(fail_list))

logger.info('total success epsisode is {}', len(success_list))
logger.info('total fail episode is {}', len(fail_list))
logger.info('number of finished at entire episode is {}', len(entire_list))

2026-05-09 01:19:31.540 | INFO     | __main__:<module>:79 - RLC-FT progress: 100/5000 episodes, success=100, fail=0


2026-05-09 01:20:20.917 | INFO     | __main__:<module>:79 - RLC-FT progress: 200/5000 episodes, success=200, fail=0


2026-05-09 01:21:12.536 | INFO     | __main__:<module>:79 - RLC-FT progress: 300/5000 episodes, success=300, fail=0


2026-05-09 01:21:59.268 | INFO     | __main__:<module>:79 - RLC-FT progress: 400/5000 episodes, success=400, fail=0


2026-05-09 01:22:44.242 | INFO     | __main__:<module>:79 - RLC-FT progress: 500/5000 episodes, success=500, fail=0


...

2026-05-09 01:56:15.444 | INFO     | __main__:<module>:79 - RLC-FT progress: 4900/5000 episodes, success=4900, fail=0


2026-05-09 01:57:04.999 | INFO     | __main__:<module>:79 - RLC-FT progress: 5000/5000 episodes, success=5000, fail=0


2026-05-09 01:57:05.001 | INFO     | __main__:<module>:81 - total success epsisode is 5000


2026-05-09 01:57:05.003 | INFO     | __main__:<module>:82 - total fail episode is 0


2026-05-09 01:57:05.005 | INFO     | __main__:<module>:83 - number of finished at entire episode is 0


In [5]:
success_list = np.array(success_list)
print('average recovery step is:')
print(np.mean(success_list[:,1]))
print(np.std(success_list[:,1]))
print('average reactive power cost is:')
print(np.mean(control_cost))
print(np.std(control_cost))
print('the total cost is:')
# print(object_cost)
print(np.mean(object_cost))
print(np.std(object_cost))

import plotly.express as px

# ---------- 3. plotting ----------
bus_idx = 2                                   # choose bus (0-based)
max_len  = max(a.shape[0] for a in voltage_trajs)

# build (episodes × max_len) matrix with NaN padding
traj_mat = np.full((len(voltage_trajs), max_len), np.nan)
for i, ep in enumerate(voltage_trajs):
    traj_mat[i, :ep.shape[0]] = ep[:, bus_idx]

mean_traj = np.nanmean(traj_mat, axis=0)
min_traj  = np.nanmin(traj_mat, axis=0)
max_traj  = np.nanmax(traj_mat, axis=0)

# pick one color from Plotly's qualitative palette
base_color = px.colors.qualitative.Plotly[0]  # '#1f77b4' (blue)

def hex_to_rgba(hex_color, alpha):
    """#1f77b4 -> rgba(31,119,180,alpha)"""
    h = hex_color.lstrip('#')
    r, g, b = tuple(int(h[i:i+2], 16) for i in (0, 2, 4))
    return f'rgba({r},{g},{b},{alpha})'

fill_color = hex_to_rgba(base_color, 0.25)    # translucent blue

fig = go.Figure()

# 1) lower bound (invisible line – starting edge of the band)
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=min_traj,
        mode="lines",
        line=dict(width=0),
        showlegend=False,
        hoverinfo="skip",
        name="min",
    )
)

# 2) upper bound with 'tonexty' fill – creates the shaded band
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=max_traj,
        mode="lines",
        line=dict(width=0),
        fill="tonexty",
        fillcolor=fill_color,
        showlegend=False,
        hoverinfo="skip",
        name="max",
    )
)

# 3) mean curve on top, same hue but solid & thicker
fig.add_trace(
    go.Scatter(
        x=np.arange(max_len),
        y=mean_traj,
        mode="lines",
        name=f"Mean Voltage (Bus {bus_idx + 1})",
        line=dict(color=base_color, width=3),
    )
)

fig.update_layout(
    title=f"Voltage Trajectories on Bus {bus_idx + 1}",
    xaxis_title="Step",
    yaxis_title="Voltage (p.u.)",
    template="plotly_white",
)

fig.show()

average recovery step is:

7.6392

7.898570969485557

average reactive power cost is:

29.993370678634726

62.92014841982708

the total cost is:

129.33833456621667

149.93081280729322

### baseline

In [6]:
### test the base line controller
num_agent = 5
voltage = []
q = []
base_cost = []
base_succ_list = []
base_fail_list = []
base_entire_list = []
base_control_cost = []
base_reward_list = []
base_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    base_cost = []
    abnormal_stop = False
    done = False

    for step in range(step_num):
        state1 = np.asarray(state-env.vmax)         # 就算 +0.001，效果也不如我们的
        state2 = np.asarray(env.vmin-state)
        d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((num_agent,1))
        
        action = (last_action - 10*d_v)
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            base_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            base_succ_list.append((episode,step))
            # Suppress per-episode success logs for large notebook runs.
            break

        voltage.append(state)

        q.append(action)

        state = next_state
        
        episode_reward += reward
        
        base_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    base_control_cost.append(episode_control)
    base_reward_list.append(episode_reward)
    base_object_cost.append(np.sum(base_cost))
    
    if (not done) and (abnormal_stop == False):
        base_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

    if (episode + 1) % 100 == 0 or (episode + 1) == episode_num:
        logger.info('Linear progress: {}/{} episodes, success={}, fail={}', episode + 1, episode_num, len(base_succ_list), len(base_fail_list))

logger.info('total success epsisode is {}', len(base_succ_list))
logger.info('total fail episode is {}', len(base_fail_list))
logger.info('number of finished at entire episode is {}', len(base_entire_list))

2026-05-09 01:58:14.972 | INFO     | __main__:<module>:72 - Linear progress: 100/5000 episodes, success=100, fail=0


2026-05-09 01:59:29.499 | INFO     | __main__:<module>:72 - Linear progress: 200/5000 episodes, success=200, fail=0


2026-05-09 02:00:42.694 | INFO     | __main__:<module>:72 - Linear progress: 300/5000 episodes, success=300, fail=0


2026-05-09 02:01:56.461 | INFO     | __main__:<module>:72 - Linear progress: 400/5000 episodes, success=400, fail=0


2026-05-09 02:03:06.855 | INFO     | __main__:<module>:72 - Linear progress: 500/5000 episodes, success=500, fail=0


...

2026-05-09 02:55:17.217 | INFO     | __main__:<module>:72 - Linear progress: 4900/5000 episodes, success=4900, fail=0


2026-05-09 02:56:28.234 | INFO     | __main__:<module>:72 - Linear progress: 5000/5000 episodes, success=5000, fail=0


2026-05-09 02:56:28.236 | INFO     | __main__:<module>:74 - total success epsisode is 5000


2026-05-09 02:56:28.238 | INFO     | __main__:<module>:75 - total fail episode is 0


2026-05-09 02:56:28.240 | INFO     | __main__:<module>:76 - number of finished at entire episode is 0


In [7]:
base_succ_list = np.array(base_succ_list)
print('average recovery step is:')
print(np.mean(base_succ_list[:,1]))
print(np.std(base_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(base_control_cost))
print(np.std(base_control_cost))
print('the total cost is:')
print(np.mean(base_object_cost))
print(np.std(base_object_cost))


average recovery step is:

38.4692

29.12178997520585

average reactive power cost is:

162.6127334507366

286.74815662603

the total cost is:

532.9865873736884

509.84231765273154

### Safe DDPG

In [8]:
### test the safe policy net
num_agent = 5
safe_voltage = []
safe_q = []
safe_cost = []
safe_succ_list = []
safe_fail_list = []
safe_entire_list = []
safe_contorl_cost = []
safe_reward_list = []
safe_object_cost = []

for episode in range(episode_num):
    state, topology, senario = env.reset_topo(seed=episode)
    topology = torch.tensor(topology, dtype=torch.float32, device=device).unsqueeze(0)
    last_action = np.zeros((agent_num,1))
    episode_reward = 0
    episode_control = 0
    safe_cost = []
    abnormal_stop = False
    done = False

    for step in range(step_num):
        action = []
        for i in range(num_agent):
            action_agent = safe_agent_net[i](torch.tensor([state[i]], dtype=torch.float32, device=device).reshape(1,1)).detach().cpu().numpy()[0]
            action.append(action_agent)
        
        action = last_action - 5*np.asarray(action).reshape((num_agent, 1))
        
        last_action = np.copy(action)
        
        try:
            next_state, reward, done = env.step(action)
        except pp.powerflow.LoadflowNotConverged:
            # logger.error(sys.exc_info())
            logger.error('power flow not converge at epsisode{} step{}', episode, step)
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break

        if(np.min(next_state) < 0.75 or np.max(next_state) > 1.25): #if voltage violation > 25%, episode ends.
            logger.warning('episode {} step {} exceeds, min_state is {}, max_state is {}', episode, step, np.min(next_state), np.max(next_state))
            safe_fail_list.append((episode,step))
            abnormal_stop = True
            break
        if done:
            safe_succ_list.append((episode,step))
            # Suppress per-episode success logs for large notebook runs.
            break
        safe_voltage.append(state)

        safe_q.append(action)

        state = next_state
        
        episode_reward += reward
        
        safe_cost.append(-reward)
        
        episode_control += LA.norm(action, 2)

    safe_contorl_cost.append(episode_control)
    safe_reward_list.append(episode_reward)
    safe_object_cost.append(np.sum(safe_cost))

    if (not done) and (abnormal_stop == False):
        safe_entire_list.append(episode)
        logger.info('Episode {} finish with entire step!', episode)

    if (episode + 1) % 100 == 0 or (episode + 1) == episode_num:
        logger.info('Safe-DDPG progress: {}/{} episodes, success={}, fail={}', episode + 1, episode_num, len(safe_succ_list), len(safe_fail_list))

logger.info('total success epsisode is {}', len(safe_succ_list))
logger.info('total fail episode is {}', len(safe_fail_list))
logger.info('number of finished at entire episode is {}', len(safe_entire_list))


2026-05-09 02:57:24.253 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 100/5000 episodes, success=100, fail=0


2026-05-09 02:58:23.953 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 200/5000 episodes, success=200, fail=0


2026-05-09 02:59:22.498 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 300/5000 episodes, success=300, fail=0


2026-05-09 03:00:22.703 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 400/5000 episodes, success=400, fail=0


2026-05-09 03:01:19.350 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 500/5000 episodes, success=500, fail=0


...

2026-05-09 03:43:03.125 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 4900/5000 episodes, success=4900, fail=0


2026-05-09 03:43:59.647 | INFO     | __main__:<module>:72 - Safe-DDPG progress: 5000/5000 episodes, success=5000, fail=0


2026-05-09 03:43:59.650 | INFO     | __main__:<module>:74 - total success epsisode is 5000


2026-05-09 03:43:59.651 | INFO     | __main__:<module>:75 - total fail episode is 0


2026-05-09 03:43:59.653 | INFO     | __main__:<module>:76 - number of finished at entire episode is 0


In [9]:
safe_succ_list = np.array(safe_succ_list)
print('average recovery step is:')
print(np.mean(safe_succ_list[:,1]))
print(np.std(safe_succ_list[:,1]))
print('average reactive power cost is:')
print(np.mean(safe_contorl_cost))
print(np.std(safe_contorl_cost))
print('the total cost is:')
print(np.mean(safe_object_cost))
print(np.std(safe_object_cost))

average recovery step is:

23.467

21.861182744764747

average reactive power cost is:

106.40349823575703

199.0305289322736

the total cost is:

325.97902979144527

370.92331964114504

In [10]:
import plotly.graph_objects as go
import numpy as np
from plotly.subplots import make_subplots

# Calculate statistics from raw data
methods = ['Linear', 'Safe-DDPG', 'RLC-FT']
metrics = ['Voltage Recovery Time', 'Reactive Power', 'Total Objective Cost']

# Extract data and calculate statistical values from the latest simulation results
base_succ_arr = np.asarray(base_succ_list)
safe_succ_arr = np.asarray(safe_succ_list)
rlc_succ_arr = np.asarray(success_list)

data = {
    'Voltage Recovery Time': {
        'means': [
            np.mean(base_succ_arr[:, 1]),      # Linear
            np.mean(safe_succ_arr[:, 1]),      # Safe-DDPG
            np.mean(rlc_succ_arr[:, 1])        # RLC-FT
        ],
        'stds': [
            np.std(base_succ_arr[:, 1]),       # Linear
            np.std(safe_succ_arr[:, 1]),       # Safe-DDPG
            np.std(rlc_succ_arr[:, 1])         # RLC-FT
        ],
        'unit': 'Steps'
    },
    'Reactive Power': {
        'means': [
            np.mean(base_control_cost),        # Linear
            np.mean(safe_contorl_cost),        # Safe-DDPG (note original variable name spelling)
            np.mean(control_cost)              # RLC-FT
        ],
        'stds': [
            np.std(base_control_cost),         # Linear
            np.std(safe_contorl_cost),         # Safe-DDPG
            np.std(control_cost)               # RLC-FT
        ],
        'unit': 'MVar'
    },
    'Total Objective Cost': {
        'means': [
            np.mean(base_object_cost),         # Linear
            np.mean(safe_object_cost),         # Safe-DDPG
            np.mean(object_cost)               # RLC-FT
        ],
        'stds': [
            np.std(base_object_cost),          # Linear
            np.std(safe_object_cost),          # Safe-DDPG
            np.std(object_cost)                # RLC-FT
        ],
        'unit': ''
    }
}

# data = {
#     'Voltage Recovery Time': {
#         'means': [
#             106.38,      # Linear
#             51.59,      # Safe-DDPG
#             17.99         # RLC-FT
#         ],
#         'stds': [
#             38.84,       # Linear
#             23.62,       # Safe-DDPG
#             12.10          # RLC-FT
#         ],
#         'unit': 'Steps'
#     },
#     'Reactive Power': {
#         'means': [
#             2034.35,        # Linear
#             1087.41,        # Safe-DDPG (note original variable name spelling)
#             271.49              # RLC-FT
#         ],
#         'stds': [
#             1696.93,         # Linear
#             1017.19,         # Safe-DDPG
#             287.73               # RLC-FT
#         ],
#         'unit': 'MVar'
#     },
#     'Total Objective Cost': {
#         'means': [
#             320150.54,         # Linear
#             177025.43,         # Safe-DDPG
#             50326.50               # RLC-FT
#         ],
#         'stds': [
#             183398.67,          # Linear
#             101214.86,          # Safe-DDPG
#             28921.66                # RLC-FT
#         ],
#         'unit': ''
#     }
# }

# Print calculated results for verification (including truncation info)
print("Calculated Statistics:")
truncation_info = []
for metric in metrics:
    print(f"\n{metric}:")
    for i, method in enumerate(methods):
        mean_val = data[metric]['means'][i]
        std_val = data[metric]['stds'][i]
        lower_bound = mean_val - std_val
        
        if lower_bound < 0:
            truncation_info.append(f"{metric} - {method}")
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f} (truncated at 0)")
        else:
            print(f"  {method}: {mean_val:.2f} ± {std_val:.2f}")

# Nature-style colors
# colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
# colors = ['#0072B2', '#D55E00', '#009E73']
# Softer Okabe–Ito palette (lighter fills)
colors = {
    'Linear':  '#5AA1D6',  # lighter blue from #0072B2
    'Safe-DDPG': '#E7904B',# lighter orange from #D55E00
    'RLC-FT':  '#53C19E'   # lighter green from #009E73
}
colors_edge = {
    'Linear':  '#2F6E9D',
    'Safe-DDPG': '#B3611E',
    'RLC-FT':  '#2D8C73'
}
err_color = 'rgba(0,0,0,0.55)'  # unified error bar color

# Create subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[metric for metric in metrics],  # Only metric names, no units
    horizontal_spacing=0.15
)

# Create grouped bar charts for each metric
for i, metric in enumerate(metrics):
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    for j, method in enumerate(methods):
        mean_val = means[j]
        std_val = stds[j]
        
        # Check if truncation is needed (handle negative values)
        is_truncated = (mean_val - std_val) < 0
        
        # Calculate error bar lengths
        if is_truncated:
            # Truncation handling: lower error bar only extends to 0
            error_minus = mean_val  # distance from mean to 0
            error_plus = std_val    # keep original upper error bar
        else:
            # Normal case: symmetric error bars
            error_minus = std_val
            error_plus = std_val
        
        bar_color = colors[method]
        edge_color = colors_edge[method]

        fig.add_trace(
            go.Bar(
                x=[method],
                y=[mean_val],
                error_y=dict(
                    type='data',
                    symmetric=False,
                    array=[error_plus],
                    arrayminus=[error_minus],
                    visible=True,
                    thickness=2.5,
                    width=5,
                    color=err_color
                ),
                name=method,
                marker_color=bar_color,
                marker_line=dict(width=1.6, color=edge_color),
                showlegend=(i == 0),
                width=0.6
            ),
            row=1, col=i+1

        )
        
        # Add value labels above error bars
        fig.add_annotation(
            x=method,
            y=mean_val + max(means) * 0.1,  # above error bars
            xshift=30,  # shift value labels to the right (in pixels, adjustable)
            text=f"{mean_val:.1f}",
            showarrow=False,
            font=dict(size=16, color='black'),
            row=1, col=i+1
        )

        # Add value labels above error bars
        # if i == 0:  #
        #     fig.add_annotation(
        #         x=method,
        #         y=mean_val + max(means) * 0.05,  # above error bars
        #         xshift=25,  # shift value labels to the right (in pixels, adjustable)
        #         text=f"{mean_val:.1f}",
        #         showarrow=False,
        #         font=dict(size=14, color='black'),
        #         row=1, col=i+1
        #     )

        # if i == 1:  #
        #     fig.add_annotation(
        #         x=method,
        #         y=mean_val + max(means) * 0.08,  # above error bars
        #         xshift=32,  # shift value labels to the right (in pixels, adjustable)
        #         text=f"{mean_val:.1f}",
        #         showarrow=False,
        #         font=dict(size=14, color='black'),
        #         row=1, col=i+1
        #     )

        # if i == 2:  #
        #     fig.add_annotation(
        #         x=method,
        #         y=mean_val + max(means) * 0.08,  # above error bars
        #         xshift=45,  # shift value labels to the right (in pixels, adjustable)
        #         text=f"{mean_val:.1f}",
        #         showarrow=False,
        #         font=dict(size=14, color='black'),
        #         row=1, col=i+1
        #     )
        
        # Add truncation marker at bottom if truncated
        if is_truncated:
            fig.add_annotation(
                x=method,
                y=max(means) * 0.02,  # near bottom
                text="⊥",  # truncation symbol
                showarrow=False,
                font=dict(size=16, color='red'),
                row=1, col=i+1
            )

# Update layout (removed title)
fig.update_layout(
    width=1200,
    height=500,
    font=dict(family="Arial, sans-serif", size=20, color="black"),
    plot_bgcolor='white',
    paper_bgcolor='white',
    legend=dict(
        x=1.02, y=1,
        xanchor='left', yanchor='top',
        bgcolor='rgba(255,255,255,0.6)',
        bordercolor='rgba(0,0,0,0)',
        font=dict(size=18)
    ),
    margin=dict(l=70, r=140, t=80, b=70)  # reduced top margin since no title
)

# Update axis styles with units on y-axes
y_axis_titles = [
    data['Voltage Recovery Time']['unit'],
    data['Reactive Power']['unit'], 
    data['Total Objective Cost']['unit']
]

for i in range(1, 4):
    fig.update_xaxes(
        row=1, col=i,
        showgrid=False,
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18)
    )
    
    # Set y-axis title with unit
    y_title = y_axis_titles[i-1] if y_axis_titles[i-1] else ""
    
    fig.update_yaxes(
        row=1, col=i,
        title=dict(
            text=y_title,
            font=dict(size=18, color='black')
        ),
        showgrid=True,
        gridwidth=0.5,
        gridcolor='lightgray',
        showline=True,
        linewidth=1.5,
        linecolor='black',
        tickfont=dict(size=18),
        rangemode='tozero'  # ensure y-axis starts from 0
    )

# Update subplot title formatting (without units now)
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=20, color='black')

# Display figure
fig.show()

# Display truncation information
if truncation_info:
    print(f"\n⊥ Truncation Applied (error bars cut at zero for):")
    for info in truncation_info:
        print(f"  - {info}")
    print("Note: Red ⊥ symbols indicate where error bars were truncated at zero")

# Data integrity check
print("\n" + "="*50)
print("Data Quality Summary:")
for metric in metrics:
    means = data[metric]['means']
    stds = data[metric]['stds']
    
    print(f"\n{metric}:")
    for j, method in enumerate(methods):
        cv = (stds[j] / means[j]) * 100  # coefficient of variation
        truncated = "Yes" if (means[j] - stds[j]) < 0 else "No"
        print(f"  {method}: CV={cv:.1f}%, Truncated={truncated}")

# Save high-quality images (optional)
# fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.png"), width=1200, height=500, scale=2)
# fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison_123.pdf"), width=1200, height=500)

fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(Config.data_path, "images","56bus", "performance_comparison.pdf"), width=1200, height=500)

fig.write_image(os.path.join(Config.data_path, "images", "56bus", "performance_comparison_5000.png"), width=1200, height=500, scale=2)
fig.write_image(os.path.join(Config.data_path, "images", "56bus", "performance_comparison_5000.pdf"), width=1200, height=500)

import json

def _method_summary(method_idx, success_arr, fail_list, entire_list):
    return {
        "success_count": int(len(success_arr)),
        "fail_count": int(len(fail_list)),
        "entire_count": int(len(entire_list)),
        "recovery_mean": float(data['Voltage Recovery Time']['means'][method_idx]),
        "recovery_std": float(data['Voltage Recovery Time']['stds'][method_idx]),
        "control_mean": float(data['Reactive Power']['means'][method_idx]),
        "control_std": float(data['Reactive Power']['stds'][method_idx]),
        "object_mean": float(data['Total Objective Cost']['means'][method_idx]),
        "object_std": float(data['Total Objective Cost']['stds'][method_idx]),
    }

summary = {
    "Linear": _method_summary(0, base_succ_arr, base_fail_list, base_entire_list),
    "Safe-DDPG": _method_summary(1, safe_succ_arr, safe_fail_list, safe_entire_list),
    "RLC-FT": _method_summary(2, rlc_succ_arr, fail_list, entire_list),
}

figure_data = {
    metric: {
        "means": [float(v) for v in data[metric]["means"]],
        "stds": [float(v) for v in data[metric]["stds"]],
        "unit": data[metric]["unit"],
    }
    for metric in metrics
}

stats_path = os.path.join(Config.data_path, "images", "56bus", "performance_main_5000_stats.json")
with open(stats_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "episodes": int(episode_num),
            "step_num": int(step_num),
            "summary": summary,
            "figure_data": figure_data,
            "truncation_info": truncation_info,
        },
        f,
        indent=2,
    )

print(f"Saved 5000-scenario stats to {stats_path}")


Calculated Statistics:


Voltage Recovery Time:

  Linear: 38.47 ± 29.12

  Safe-DDPG: 23.47 ± 21.86

  RLC-FT: 7.64 ± 7.90 (truncated at 0)

...


Total Objective Cost:

  Linear: CV=95.7%, Truncated=No

  Safe-DDPG: CV=113.8%, Truncated=Yes

  RLC-FT: CV=115.9%, Truncated=Yes

Saved 5000-scenario stats to D:/Code/Python/Flexible_Voltage_Control/images\56bus\performance_main_5000_stats.json